# Análisis Exploratorio Inicial — Crime Data from 2020 to Present (LAPD)

**Objetivo:** Explorar la estructura, calidad y características del dataset para definir las decisiones de limpieza que se implementarán en un notebook posterior.

**Dataset:** [Crime Data from 2020 to Present](https://data.lacity.org/Public-Safety/Crime-Data-from-2020-to-Present/2nrs-mtv8) — Los Angeles Police Department (LAPD)



## 1. Configuración e importaciones

In [1]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

In [2]:
# Configuración de display para poder ver todas las columnas del dataset
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_colwidth', None)
pd.set_option('display.width', None)

In [3]:
df = pd.read_csv('../data/raw/Crime_Data_from_2020_to_Present.csv')


## 2. Vista general del dataset

In [4]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 1005198 entries, 0 to 1005197
Data columns (total 28 columns):
 #   Column          Non-Null Count    Dtype  
---  ------          --------------    -----  
 0   DR_NO           1005198 non-null  int64  
 1   Date Rptd       1005198 non-null  str    
 2   DATE OCC        1005198 non-null  str    
 3   TIME OCC        1005198 non-null  int64  
 4   AREA            1005198 non-null  int64  
 5   AREA NAME       1005198 non-null  str    
 6   Rpt Dist No     1005198 non-null  int64  
 7   Part 1-2        1005198 non-null  int64  
 8   Crm Cd          1005198 non-null  int64  
 9   Crm Cd Desc     1005198 non-null  str    
 10  Mocodes         853438 non-null   str    
 11  Vict Age        1005198 non-null  int64  
 12  Vict Sex        860416 non-null   str    
 13  Vict Descent    860404 non-null   str    
 14  Premis Cd       1005182 non-null  float64
 15  Premis Desc     1004610 non-null  str    
 16  Weapon Used Cd  327280 non-null   float64
 17  

**Observaciones iniciales:**
- El dataset contiene **~1,005,198 registros** y **28 columnas**.
- Tipos de datos: 7 `int64`, 8 `float64`, 13 `str`.
- Las columnas `Date Rptd` y `DATE OCC` son de tipo `str` en formato `M/D/YYYY 0:00` → **necesitan conversión a `datetime`**.
- `TIME OCC` es un entero en formato HHMM (ej: 1315 = 1:15 PM) → **considerar convertir a hora del día (0-23)**.
- Varias columnas tienen una cantidad significativa de valores nulos (analizado en sección 3).

In [23]:
df.sample(10)

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
82121,200610634,5/26/2020 0:00,5/26/2020 0:00,1055,6,Hollywood,646,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0450 1309 1822,24,M,B,102.0,SIDEWALK,500.0,UNKNOWN WEAPON/OTHER WEAPON,IC,Invest Cont,230.0,NaN,NaN,NaN,HOLLYWOOD BL,WHITLEY AV,34.1016,-118.3333
810066,230119753,9/13/2023 0:00,9/13/2023 0:00,1,1,Central,182,1,510,VEHICLE - STOLEN,NaN,0,NaN,NaN,108.0,PARKING LOT,NaN,NaN,IC,Invest Cont,510.0,NaN,NaN,NaN,1200 S FLOWER ST,NaN,34.0415,-118.2655
125510,201712867,9/3/2020 0:00,8/15/2020 0:00,800,17,Devonshire,1762,1,480,BIKE - STOLEN,344,52,M,W,501.0,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,480.0,NaN,NaN,NaN,9600 MASON AV,NaN,34.2445,-118.5798
77378,200118279,9/20/2020 0:00,9/20/2020 0:00,2035,1,Central,148,1,230,"ASSAULT WITH DEADLY WEAPON, AGGRAVATED ASSAULT",0334 0445 0319 0913 2003 0361 2004 1218,39,M,B,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",207.0,OTHER KNIFE,AO,Adult Other,230.0,NaN,NaN,NaN,600 E 5TH ST,NaN,34.0438,-118.2416
109035,200610740,5/29/2020 0:00,5/29/2020 0:00,400,6,Hollywood,659,2,745,VANDALISM - MISDEAMEANOR ($399 OR UNDER),0329 0601,0,X,X,203.0,OTHER BUSINESS,NaN,NaN,AA,Adult Arrest,745.0,NaN,NaN,NaN,5400 FOUNTAIN AV,NaN,34.0966,-118.3049
498961,221716457,9/27/2022 0:00,9/3/2022 0:00,420,17,Devonshire,1796,1,331,THEFT FROM MOTOR VEHICLE - GRAND ($950.01 AND OVER),NaN,26,F,O,101.0,STREET,NaN,NaN,IC,Invest Cont,331.0,NaN,NaN,NaN,16800 PARTHENIA ST,NaN,34.2285,-118.4983
662150,231017252,12/15/2023 0:00,12/15/2023 0:00,940,10,West Valley,1041,2,624,BATTERY - SIMPLE ASSAULT,0400 0416 1202 0913 0603,61,F,O,501.0,SINGLE FAMILY DWELLING,400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",AO,Adult Other,624.0,NaN,NaN,NaN,6000 CAHILL AV,NaN,34.1790,-118.5491
902078,241208999,3/26/2024 0:00,3/26/2024 0:00,1540,12,77th Street,1273,2,354,THEFT OF IDENTITY,100,22,M,B,501.0,SINGLE FAMILY DWELLING,NaN,NaN,IC,Invest Cont,354.0,NaN,NaN,NaN,8600 S GRAMERCY PL,NaN,33.9600,-118.3134
124831,201610102,6/15/2020 0:00,6/15/2020 0:00,1930,16,Foothill,1606,2,624,BATTERY - SIMPLE ASSAULT,0913 0416 1601 1202,63,F,H,501.0,SINGLE FAMILY DWELLING,400.0,"STRONG-ARM (HANDS, FIST, FEET OR BODILY FORCE)",AO,Adult Other,624.0,NaN,NaN,NaN,11700 TERRA BELLA ST,NaN,34.2809,-118.3874
387315,211220220,9/23/2021 0:00,9/23/2021 0:00,1325,12,77th Street,1258,1,210,ROBBERY,0344 1822 0216 0342 0449 0305,20,M,H,502.0,"MULTI-UNIT DWELLING (APARTMENT, DUPLEX, ETC)",102.0,HAND GUN,IC,Invest Cont,210.0,NaN,NaN,NaN,79TH,MAIN,33.9679,-118.2739


**Observaciones del sample:**
- La columna `LOCATION` tiene **muchos espacios extra** (padding) en sus valores → necesita normalización con `.str.strip()` y `.str.replace(r'\s+', ' ', regex=True)`.
- Se observan registros con `Vict Age = 0`, `Vict Sex = NaN/X` y `Vict Descent = NaN/X`, lo que sugiere crímenes sin víctima identificada.

### 2.1 Diccionario de columnas

In [6]:
columnas_descripciones = {
    'DR_NO': 'Unique identifier for the crime report',
    'Date Rptd': 'The date the crime was reported',
    'DATE OCC': 'The actual date the crime occurred',
    'TIME OCC': 'The time the crime occurred, in HHMM 24-hour format',
    'AREA': 'Numeric code representing the geographical area (1-21)',
    'AREA NAME': 'Name of the geographical area',
    'Rpt Dist No': 'Reporting district number for the incident',
    'Part 1-2': 'Crime classification (Part 1 = serious crimes, Part 2 = less serious)',
    'Crm Cd': 'Numeric code representing the type of crime',
    'Crm Cd Desc': 'Description of the crime type',
    'Mocodes': 'Modus operandi codes, describing the method used in the crime',
    'Vict Age': 'Age of the victim (0 = unknown/not applicable)',
    'Vict Sex': 'Gender of the victim (M, F, X=Unknown, H, -)',
    'Vict Descent': 'Ethnicity or descent of the victim',
    'Premis Cd': 'Numeric code for the type of premises',
    'Premis Desc': 'Description of the type of premises',
    'Weapon Used Cd': 'Numeric code for the weapon used (NaN = no weapon)',
    'Weapon Desc': 'Description of the weapon used',
    'Status': 'Status code of the crime case',
    'Status Desc': 'Description of the case status',
    'Crm Cd 1': 'Primary crime code (nearly identical to Crm Cd)',
    'Crm Cd 2': 'Secondary crime code (if multiple offenses)',
    'Crm Cd 3': 'Tertiary crime code (if multiple offenses)',
    'Crm Cd 4': 'Quaternary crime code (if multiple offenses)',
    'LOCATION': 'Text description of the crime location (has extra whitespace)',
    'Cross Street': 'Nearby cross street for the crime location',
    'LAT': 'Latitude of the crime location (0.0 = unknown)',
    'LON': 'Longitude of the crime location (0.0 = unknown)'
}

df_columnas = pd.DataFrame(list(columnas_descripciones.items()), columns=['columna', 'descripcion'])
df_columnas.to_csv('../data/raw/descripciones_columnas.csv', index=False)
df_columnas

,columna,descripcion
0,DR_NO,Unique identifier for the crime report
1,Date Rptd,The date the crime was reported
2,DATE OCC,The actual date the crime occurred
3,TIME OCC,"The time the crime occurred, in HHMM 24-hour format"
4,AREA,Numeric code representing the geographical area (1-21)
...,...,...
23,Crm Cd 4,Quaternary crime code (if multiple offenses)
24,LOCATION,Text description of the crime location (has extra whitespace)
25,Cross Street,Nearby cross street for the crime location
26,LAT,Latitude of the crime location (0.0 = unknown)



## 3. Valores nulos y duplicados

### 3.1 Filas duplicadas

In [7]:
print(f"Filas duplicadas: {df.duplicated().sum()}")

Filas duplicadas: 0


**No hay filas duplicadas** en el dataset.

### 3.2 Valores nulos por columna

In [8]:
null_counts = df.isnull().sum()
null_pct = (null_counts / len(df) * 100).round(2)

null_summary = pd.DataFrame({
    'nulos': null_counts,
    'porcentaje': null_pct
}).sort_values('nulos', ascending=False)

null_summary[null_summary['nulos'] > 0]

,nulos,porcentaje
Crm Cd 4,1005134,99.99
Crm Cd 3,1002884,99.77
Crm Cd 2,936039,93.12
Cross Street,850955,84.66
Weapon Used Cd,677918,67.44
...,...,...
Vict Sex,144782,14.40
Premis Desc,588,0.06
Premis Cd,16,0.00
Crm Cd 1,11,0.00


**Análisis de nulos:**

| Grupo | Columnas | Nulos | Acción sugerida |
|---|---|---|---|
| **Extremadamente dispersas** | `Crm Cd 4` (99.99%), `Crm Cd 3` (99.8%), `Crm Cd 2` (93.1%) | 936K-1M+ | **Eliminar** — representan crímenes secundarios, casi nunca se usan |
| **Muy alta** | `Cross Street` (84.7%) | 851K | **Eliminar** — demasiados nulos para ser útil |
| **Esperada (sin arma)** | `Weapon Used Cd`, `Weapon Desc` (67.4%) | 678K | **Imputar** `Weapon Desc` NaN → `"NO WEAPON"` — nulo es válido (no hubo arma) |
| **Sin víctima** | `Mocodes` (15.1%), `Vict Sex` (14.4%), `Vict Descent` (14.4%) | 145K-152K | **Tratar** como parte del problema de víctimas no identificadas |
| **Mínimos** | `Premis Desc` (588), `Premis Cd` (16), `Crm Cd 1` (11), `Status` (1) | <600 | **Ignorar** o imputar puntualmente |


## 4. Cardinalidad y pares redundantes

In [24]:
cardinalidad_df = pd.DataFrame({
    'columna': df.columns,
    'tipo': df.dtypes.values,
    'cardinalidad': df.nunique().values,
    'total_registros': len(df),
    'porcentaje_cardinalidad': (df.nunique() / len(df) * 100).values
})
pd.set_option('display.max_rows', None)
cardinalidad_df.sort_values(by='cardinalidad', ascending=True)

,columna,tipo,cardinalidad,total_registros,porcentaje_cardinalidad
7,Part 1-2,int64,2,1005198,0.000199
12,Vict Sex,str,5,1005198,0.000497
18,Status,str,6,1005198,0.000597
19,Status Desc,str,6,1005198,0.000597
23,Crm Cd 4,float64,6,1005198,0.000597
13,Vict Descent,str,20,1005198,0.001990
5,AREA NAME,str,21,1005198,0.002089
4,AREA,int64,21,1005198,0.002089
22,Crm Cd 3,float64,38,1005198,0.003780
16,Weapon Used Cd,float64,79,1005198,0.007859


**Pares redundantes identificados (código numérico ↔ descripción textual):**

En cada par, el código y la descripción representan exactamente la misma información. Para simplificar el dataset, se conservará **solo la descripción textual** y se eliminará el código numérico.

| Código | Descripción | Card. código | Card. desc. | Conservar |
|---|---|---|---|---|
| `AREA` | `AREA NAME` | 21 | 21 | `AREA NAME` |
| `Crm Cd` | `Crm Cd Desc` | 140 | 140 | `Crm Cd Desc` |
| `Premis Cd` | `Premis Desc` | 314 | 306 | `Premis Desc` |
| `Weapon Used Cd` | `Weapon Desc` | 79 | 79 | `Weapon Desc` |
| `Status` | `Status Desc` | 6 | 6 | `Status Desc` |

**Otras columnas a eliminar:**
- `DR_NO` — Identificador único, no aporta valor analítico.
- `Rpt Dist No` — Granularidad innecesaria si ya tenemos `AREA NAME`.
- `Crm Cd 1` — Casi idéntico a `Crm Cd` (142 vs 140 únicos). Redundante.
- `Mocodes` — Cardinalidad extrema (310,956 combinaciones, 31%). Difícil de usar sin parsing complejo. **Se elimina.**


## 5. Análisis de variables numéricas

In [10]:
df.describe()

,DR_NO,TIME OCC,AREA,Rpt Dist No,Part 1-2,Crm Cd,Vict Age,Premis Cd,Weapon Used Cd,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LAT,LON
count,1.005198e+06,1.005198e+06,1.005198e+06,1.005198e+06,1.005198e+06,1.005198e+06,1.005198e+06,1.005182e+06,327280.000000,1.005187e+06,69159.000000,2314.000000,64.00000,1.005198e+06,1.005198e+06
mean,2.202277e+08,1.339911e+03,1.069098e+01,1.115556e+03,1.400283e+00,5.001458e+02,2.891253e+01,3.056189e+02,363.953651,4.999063e+02,958.105221,984.015990,991.21875,3.399820e+01,-1.180909e+02
std,1.320282e+07,6.510531e+02,6.110385e+00,6.111733e+02,4.899559e-01,2.052635e+02,2.199382e+01,2.193160e+02,123.736081,2.050640e+02,110.354136,52.350982,27.06985,1.610549e+00,5.581812e+00
min,8.170000e+02,1.000000e+00,1.000000e+00,1.010000e+02,1.000000e+00,1.100000e+02,-4.000000e+00,1.010000e+02,101.000000,1.100000e+02,210.000000,310.000000,821.00000,0.000000e+00,-1.186676e+02
25%,2.106169e+08,9.000000e+02,5.000000e+00,5.870000e+02,1.000000e+00,3.310000e+02,0.000000e+00,1.010000e+02,311.000000,3.310000e+02,998.000000,998.000000,998.00000,3.401470e+01,-1.184305e+02
50%,2.209160e+08,1.420000e+03,1.100000e+01,1.139000e+03,1.000000e+00,4.420000e+02,3.000000e+01,2.030000e+02,400.000000,4.420000e+02,998.000000,998.000000,998.00000,3.405890e+01,-1.183225e+02
75%,2.311105e+08,1.900000e+03,1.600000e+01,1.613000e+03,2.000000e+00,6.260000e+02,4.400000e+01,5.010000e+02,400.000000,6.260000e+02,998.000000,998.000000,998.00000,3.416490e+01,-1.182739e+02
max,2.521041e+08,2.359000e+03,2.100000e+01,2.199000e+03,2.000000e+00,9.560000e+02,1.200000e+02,9.760000e+02,516.000000,9.560000e+02,999.000000,999.000000,999.00000,3.433430e+01,0.000000e+00


### 5.1 Análisis detallado de `Vict Age`

La columna de edad muestra anomalías claras: el mínimo es **-4**, la moda es **0**, y el percentil 25 es **0**. Esto indica que el valor `0` se usa como placeholder cuando la víctima no fue identificada o el crimen no tiene víctima directa.

In [11]:
print("=== ESTADÍSTICAS DE EDAD ===")
print(df['Vict Age'].describe())
print(f"\nMediana: {df['Vict Age'].median()}")
print(f"Moda: {df['Vict Age'].mode().values[0]}")
print(f"Rango: {df['Vict Age'].min()} - {df['Vict Age'].max()}")

=== ESTADÍSTICAS DE EDAD ===
count    1.005198e+06
mean     2.891253e+01
std      2.199382e+01
min     -4.000000e+00
25%      0.000000e+00
50%      3.000000e+01
75%      4.400000e+01
max      1.200000e+02
Name: Vict Age, dtype: float64

Mediana: 30.0
Moda: 0
Rango: -4 - 120


In [12]:
# Registros con edades inválidas (≤0 o >100)
edades_incorrectas = df[(df['Vict Age'] <= 0) | (df['Vict Age'] > 100)]
print(f"Total de registros con edad inválida (≤0 o >100): {len(edades_incorrectas):,}")
print(f"  - Con edad = 0: {len(df[df['Vict Age'] == 0]):,}")
print(f"  - Con edad < 0: {len(df[df['Vict Age'] < 0]):,}")
print(f"  - Con edad > 100: {len(df[df['Vict Age'] > 100]):,}")

Total de registros con edad inválida (≤0 o >100): 269,514
  - Con edad = 0: 269,377
  - Con edad < 0: 136
  - Con edad > 100: 1


In [13]:
# Muestra de registros con edad = 0: se observa que coinciden con Vict Sex = NaN y Vict Descent = NaN
# Son crímenes sin víctima directa (robo de vehículo, vandalismo, etc.)
df[df['Vict Age'] == 0].head()

,DR_NO,Date Rptd,DATE OCC,TIME OCC,AREA,AREA NAME,Rpt Dist No,Part 1-2,Crm Cd,Crm Cd Desc,Mocodes,Vict Age,Vict Sex,Vict Descent,Premis Cd,Premis Desc,Weapon Used Cd,Weapon Desc,Status,Status Desc,Crm Cd 1,Crm Cd 2,Crm Cd 3,Crm Cd 4,LOCATION,Cross Street,LAT,LON
0,190326475,3/1/2020 0:00,3/1/2020 0:00,2130,7,Wilshire,784,1,510,VEHICLE - STOLEN,NaN,0,M,O,101.0,STREET,NaN,NaN,AA,Adult Arrest,510.0,998.0,NaN,NaN,1900 S LONGWOOD AV,NaN,34.0375,-118.3506
4,200412582,9/9/2020 0:00,9/9/2020 0:00,630,4,Hollenbeck,413,1,510,VEHICLE - STOLEN,NaN,0,NaN,NaN,101.0,STREET,NaN,NaN,IC,Invest Cont,510.0,NaN,NaN,NaN,200 E AVENUE 28,NaN,34.0820,-118.2130
5,200209713,5/3/2020 0:00,5/2/2020 0:00,1800,2,Rampart,245,1,510,VEHICLE - STOLEN,NaN,0,NaN,NaN,101.0,STREET,NaN,NaN,IC,Invest Cont,510.0,NaN,NaN,NaN,2500 W 4TH ST,NaN,34.0642,-118.2771
6,200200759,7/7/2020 0:00,7/7/2020 0:00,1340,2,Rampart,265,1,648,ARSON,0329 1402 2004 1501,0,X,X,101.0,STREET,NaN,NaN,IC,Invest Cont,648.0,998.0,NaN,NaN,JAMES M WOOD,ALVARADO,34.0536,-118.2788
7,201308739,3/27/2020 0:00,3/27/2020 0:00,1210,13,Newton,1333,1,510,VEHICLE - STOLEN,NaN,0,NaN,NaN,101.0,STREET,NaN,NaN,IC,Invest Cont,510.0,NaN,NaN,NaN,3200 S SAN PEDRO ST,NaN,34.0170,-118.2643


**Conclusión:** Los 269,377 registros con `Vict Age = 0` y los 136 con edad negativa son valores centinela que representan víctimas no identificadas o crímenes sin víctima. **Acción: reemplazar `Vict Age ≤ 0` y `> 99` por `NaN`.**


## 6. Análisis de variables categóricas

### 6.1 Sexo de la víctima (`Vict Sex`)

In [14]:
print("Distribución de Vict Sex:")
print(df['Vict Sex'].value_counts(dropna=False))
print(f"\nTotal de registros con sexo != M/F: {len(df[(df['Vict Sex'] != 'M') & (df['Vict Sex'] != 'F')]):,}")

Distribución de Vict Sex:
Vict Sex
M      403916
F      358599
NaN    144782
X       97786
H         114
-           1
Name: count, dtype: int64

Total de registros con sexo != M/F: 242,683


**Valores encontrados:**
- `M` (Masculino) y `F` (Femenino) → valores válidos.
- `X` (~97K) → desconocido / no aplica.
- `H` (114) → probablemente un error de entrada. 
- `-` (1) → error claro.
- `NaN` (~145K) → sin dato.

**Acción:** Unificar `H`, `-`, `X` y `NaN` como un único valor `X/Unknown` (desconocido/no aplica).

In [15]:
# Desglose de valores de sexo en registros con sexo != M/F (sin contar NaN)
sexos_no_mf = df[(df['Vict Sex'] != 'M') & (df['Vict Sex'] != 'F')]
sexos_no_mf['Vict Sex'].value_counts(dropna=False)

Vict Sex
NaN    144782
X       97786
H         114
-           1
Name: count, dtype: int64

### 6.2 Estado del caso (`Status`)

In [16]:
df['Status'].value_counts()

Status
IC    803946
AO    109255
AA     86877
JA      3249
JO      1864
CC         6
Name: count, dtype: int64

**Valores:**
- `IC` (Invest Cont) = Investigación en curso — 80% de los casos.
- `AO` (Adult Other) = 10.9%
- `AA` (Adult Arrest) = 8.6%
- `JA`, `JO`, `CC` = porcentajes mínimos.

Sin problemas de calidad.

### 6.3 Víctimas no identificadas

Muchos registros comparten el patrón: `Vict Age = 0`, `Vict Sex ≠ M/F`, `Vict Descent = X`. Estos representan crímenes donde la víctima no fue identificada o no aplica (ej: robo de vehículo, vandalismo).

In [17]:
victimas_no_identificadas = df[
    (df['Vict Sex'] != 'M') & 
    (df['Vict Sex'] != 'F') & 
    (df['Vict Age'] <= 0) & 
    (df['Vict Descent'] == 'X')
]
print(f"Registros con víctima no identificada (edad≤0, sexo≠M/F, descent=X): {len(victimas_no_identificadas):,}")
print(f"Esto representa el {len(victimas_no_identificadas)/len(df)*100:.1f}% del dataset.")

Registros con víctima no identificada (edad≤0, sexo≠M/F, descent=X): 87,233
Esto representa el 8.7% del dataset.


**Acción:** Crear un flag booleano `victim_identified` para marcar estos ~87K registros. Esto permite filtrarlos fácilmente en análisis posteriores sin perder datos.

**Sobre `Vict Descent`:** Los valores `X` representan descendencia desconocida → convertir a `NaN` durante la limpieza.


## 7. Análisis geoespacial y de ubicación

### 7.1 Coordenadas inválidas (LAT=0, LON=0)

In [18]:
invalid_coords = df[(df['LAT'] == 0) | (df['LON'] == 0)]
print(f"Registros con LAT = 0: {len(df[df['LAT'] == 0]):,}")
print(f"Registros con LON = 0: {len(df[df['LON'] == 0]):,}")
print(f"Total con coordenadas inválidas: {len(invalid_coords):,} ({len(invalid_coords)/len(df)*100:.2f}%)")

Registros con LAT = 0: 2,240
Registros con LON = 0: 2,240
Total con coordenadas inválidas: 2,240 (0.22%)


In [19]:
# ¿Los registros con coordenadas inválidas tienen AREA NAME? → Sí, todos
print(f"Registros sin coordenadas que SÍ tienen AREA NAME: {len(invalid_coords) - invalid_coords['AREA NAME'].isnull().sum()}/{len(invalid_coords)}")
print("\nDistribución por área:")
invalid_coords['AREA NAME'].value_counts()

Registros sin coordenadas que SÍ tienen AREA NAME: 2240/2240

Distribución por área:


AREA NAME
Hollywood      322
Central        194
Pacific        173
Southeast      141
77th Street    133
              ... 
Harbor          61
West LA         47
Northeast       44
West Valley     37
Topanga         30
Name: count, Length: 21, dtype: int64

**Hallazgo:** Los 2,240 registros con coordenadas `(0, 0)` sí tienen `AREA NAME` asignado. Esto permite imputar las coordenadas usando la **mediana de LAT/LON** del área correspondiente.

**Acción:** Reemplazar `LAT = 0` y `LON = 0` por `NaN`, y considerar imputación por mediana del `AREA NAME`.

### 7.2 Limpieza de `LOCATION`

In [20]:
# Los valores de LOCATION tienen padding con espacios extra
df['LOCATION'].sample(5)

196406                           11200 S  BROADWAY
636654    11100    PEORIA                       ST
669259    11900    FOOTHILL                     BL
7434        500 S  SOTO                         ST
979001             36TH                         ST
Name: LOCATION, dtype: str

**Acción:** Aplicar `.str.strip()` y normalizar espacios múltiples con `.str.replace(r'\s+', ' ', regex=True)` a `LOCATION`.


## 8. Resumen de decisiones de limpieza

A continuación se consolidan todas las decisiones de limpieza identificadas en este análisis.

### 8.1 Columnas a eliminar

| Columna | Motivo |
|---|---|
| `DR_NO` | Identificador único, sin valor analítico |
| `Rpt Dist No` | Granularidad innecesaria (ya tenemos `AREA NAME`) |
| `Mocodes` | Cardinalidad extrema (~31%), difícil de usar |
| `Crm Cd 2` | 93.1% nulos |
| `Crm Cd 3` | 99.8% nulos |
| `Crm Cd 4` | 99.99% nulos |
| `Cross Street` | 84.7% nulos |

### 8.2 Conversiones de tipo

| Columna | De | A |
|---|---|---|
| `Date Rptd` | `str` | `datetime` (con `pd.to_datetime()`) |
| `DATE OCC` | `str` | `datetime` (con `pd.to_datetime()`) |
| `TIME OCC` | `int` (HHMM) | Extraer hora (0-23): `TIME OCC // 100` |

### 8.3 Tratamiento de valores centinela

| Columna | Condición | Acción |
|---|---|---|
| `Vict Age` | `≤ 0` o `> 99` | → `NaN` |
| `Vict Sex` | `H`, `-`, `X`, `NaN` | → Unificar como `X/Unknown` |
| `Vict Descent` | `X` | → `NaN` |
| `LAT` | `= 0` | → `NaN` (imputar por mediana del `AREA NAME`) |
| `LON` | `= 0` | → `NaN` (imputar por mediana del `AREA NAME`) |
| `Weapon Desc` | `NaN` | → `"NO WEAPON"` |

### 8.4 Limpieza de strings 

| Columna | Acción |
|---|---|
| `LOCATION` | `.str.strip()` + normalizar espacios múltiples |
| Columnas de texto en general | Verificar consistencia |

### 8.5 Feature engineering (post-limpieza)

| Feature nueva | Fuente | Descripción |
|---|---|---|
| `year` | `DATE OCC` | Año de ocurrencia |
| `month` | `DATE OCC` | Mes de ocurrencia |
| `day_of_week` | `DATE OCC` | Día de la semana |
| `hour` | `TIME OCC` | Hora del día (0-23) |
| `days_to_report` | `Date Rptd - DATE OCC` | Días entre ocurrencia y reporte |
| `victim_identified` | `Vict Age`, `Vict Sex`, `Vict Descent` | Flag booleano para filtrar crímenes sin víctima |

### 8.6 Renombrar columnas

Estandarizar todos los nombres a **snake_case** (minúsculas, guiones bajos, sin espacios).